# Demo 6：HD 视频的网络损伤模拟

**课程**：未来媒体互联网
**时长**：约 10 分钟
**环境**：Kaggle Notebook（CPU，NumPy + Matplotlib + SciPy）

## 实验目标

模拟丢包、抖动和带宽骤降对 HD 视频画质的影响，
展示不同网络损伤类型如何产生视觉上各异的效果。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
print('Imports OK')


In [ ]:
# Create synthetic HD test frame (scaled to 540x960)
H, W = 540, 960

def create_test_frame():
    frame = np.zeros((H, W, 3), dtype=np.float32)
    colors = [(1,0,0),(0,1,0),(0,0,1),(1,1,0),(0,1,1),(1,0,1),(1,1,1)]
    bar_w = W // len(colors)
    for i, c in enumerate(colors):
        frame[:H//3, i*bar_w:(i+1)*bar_w] = c
    for x in range(W):
        frame[H//3:2*H//3, x] = (x/W, 0.5, 1-x/W)
    for y in range(2*H//3, H, 20):
        for x in range(0, W, 4):
            v = 0.9 if (x//4+y//20)%2==0 else 0.2
            frame[y:y+10, x:x+2] = v
    return frame

original = create_test_frame()
fig, ax = plt.subplots(figsize=(12,7))
ax.imshow(np.clip(original,0,1))
ax.set_title('Original HD Test Frame (540x960)')
ax.axis('off')
plt.show()


In [ ]:
# Network impairment simulators (frame-size-aware)
def packet_loss(frame, rate):
    result = frame.copy()
    Hf, Wf = frame.shape[:2]
    bh, bw = 30, 40
    nb_h, nb_w = Hf//bh, Wf//bw
    mask = np.random.random((nb_h, nb_w)) < rate
    for i in range(nb_h):
        for j in range(nb_w):
            if mask[i,j]:
                i1, i2 = i*bh, min((i+1)*bh, Hf)
                j1, j2 = j*bw, min((j+1)*bw, Wf)
                result[i1:i2, j1:j2] = 0.5
    return result

def jitter(frame, px):
    result = frame.copy()
    Hf = frame.shape[0]
    shifts = (np.random.randn(Hf)*px).astype(int)
    for y in range(Hf):
        result[y] = np.roll(result[y], shifts[y], axis=0)
    return result

def bw_drop(frame, scale):
    Hf, Wf = frame.shape[:2]
    h2, w2 = int(Hf*scale), int(Wf*scale)
    low = ndimage.zoom(frame, (scale,scale,1), order=1)
    up = ndimage.zoom(low, (1/scale,1/scale,1), order=1)
    return np.clip(up[:Hf,:Wf], 0, 1)

print("Simulators ready (frame-size-aware)")


In [ ]:
# Full comparison grid
np.random.seed(42)
loss_rates = [0.01, 0.05, 0.10, 0.20]
jitter_px = [1, 3, 6, 10]
bw_scales = [0.75, 0.50, 0.25, 0.10]

fig, axes = plt.subplots(3, 5, figsize=(18, 12))

for row, (label, levels, func) in enumerate([
    ('Packet Loss', loss_rates, lambda f,l: packet_loss(f,l)),
    ('Jitter', jitter_px, lambda f,l: jitter(f,l)),
    ('BW Drop', bw_scales, lambda f,l: bw_drop(f,l)),
]):
    axes[row,0].imshow(original)
    axes[row,0].set_title('Original', fontweight='bold')
    for j, lv in enumerate(levels):
        imp = func(original, lv)
        axes[row,j+1].imshow(np.clip(imp,0,1))
        axes[row,j+1].set_title('{} {}'.format(label, lv))
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Network Impairments on HD Video', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Detail zoom comparison
dy, dx = slice(200,350), slice(400,550)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

axes[0,0].imshow(original[dy,dx])
axes[0,0].set_title('Original Detail', fontweight='bold')
axes[0,0].axis('off')

pl = packet_loss(original, 0.10)
axes[0,1].imshow(np.clip(pl[dy,dx],0,1))
axes[0,1].set_title('Packet Loss 10%')
axes[0,1].axis('off')

jt = jitter(original, 6)
axes[0,2].imshow(np.clip(jt[dy,dx],0,1))
axes[0,2].set_title('Jitter 6px')
axes[0,2].axis('off')

bw = bw_drop(original, 0.25)
axes[1,0].imshow(np.clip(bw[dy,dx],0,1))
axes[1,0].set_title('BW Drop 25%')
axes[1,0].axis('off')

cb = packet_loss(bw_drop(original, 0.5), 0.05)
axes[1,1].imshow(np.clip(cb[dy,dx],0,1))
axes[1,1].set_title('BW 50% + Loss 5%')
axes[1,1].axis('off')

cb2 = jitter(packet_loss(original, 0.05), 3)
axes[1,2].imshow(np.clip(cb2[dy,dx],0,1))
axes[1,2].set_title('Loss 5% + Jitter 3px')
axes[1,2].axis('off')

plt.suptitle('Detail Zoom: Artifact Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 关键结论

- **丢包**：丢失的宏块（灰色方块）。错误隐藏有帮助，但会留下可见伪影。
- **抖动**：水平撕裂（线条错位）。对于无缓冲的直播场景尤为致命。
- **带宽骤降**：强制降分辨率导致整体模糊。
- **组合损伤**：复合伪影比单一损伤更严重。

## 实际缓解方案
- YouTube/Netflix：ABR（自适应码率）+ FEC（前向纠错）+ 抖动缓冲
- WebRTC/Google Meet：NACK（重传）+ FEC + 自适应编码
- 5G URLLC：超低时延降低抖动敏感性


In [ ]:
# 模拟真实视频场景：自然风景 + 文字叠加（类新闻画面）
def create_realistic_scene():
    H, W = 360, 640
    frame = np.zeros((H, W, 3), dtype=np.float32)
    # 天空渐变
    for y in range(H//2):
        t = y / (H//2)
        frame[y, :] = (0.3+0.4*t, 0.4+0.3*t, 0.6+0.3*t)
    # 远山轮廓
    import random; random.seed(42)
    for x in range(W):
        h = int(H*0.35 + np.sin(x*0.01)*20 + np.sin(x*0.03)*15)
        frame[h:H//2+30, x] = (0.15, 0.35, 0.15)
    # 地面纹理
    for y in range(H//2+30, H):
        v = 0.2 + 0.15 * np.sin(y*0.05) * np.sin(y*0.02)
        frame[y, :] = (v, 0.25+v*0.3, v*0.5)
    # 建筑群
    buildings = [(50,80,120,0.6),(180,50,200,0.5),(320,100,150,0.45),(450,70,170,0.55)]
    for bx, bw, bh, c in buildings:
        top = H//2+30 - bh
        frame[top:H//2+30, bx:bx+bw] = (c*0.7, c*0.6, c*0.5)
        # 窗户
        for wy in range(top+5, top+bh-5, 15):
            for wx in range(bx+5, bx+bw-5, 12):
                frame[wy:wy+8, wx:wx+6] = (0.8, 0.85, 0.7) if (wx+wy)%30<15 else (0.3, 0.3, 0.2)
    # 文字叠加（模拟新闻标题栏）
    frame[H-40:H-10, 20:W-20] = (0.05, 0.05, 0.1)
    frame[H-35:H-15, 30:W-30] = (0.9, 0.9, 0.9)
    return frame

scene = create_realistic_scene()
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(np.clip(scene, 0, 1))
ax.set_title('Simulated Video Scene (Landscape + Buildings + Text Overlay)')
ax.axis('off')
plt.show()


In [ ]:
# 在真实感场景上应用网络损伤
np.random.seed(123)
impairments_real = [
    ('Original', lambda f: f),
    ('Loss 10%', lambda f: packet_loss(f, 0.10)),
    ('Jitter 6px', lambda f: jitter(f, 6)),
    ('BW Drop 30%', lambda f: bw_drop(f, 0.30)),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for j, (label, func) in enumerate(impairments_real):
    imp = func(scene)
    axes[j].imshow(np.clip(imp, 0, 1))
    axes[j].set_title(label, fontsize=12, fontweight='bold')
    axes[j].axis('off')
plt.suptitle('Realistic Scene: Network Impairment Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n在实际视频传输中，网络损伤的影响体现在：")
print("1. 丢包 -> 画面中出现灰色方块（错误隐藏后的残留）")
print("2. 抖动 -> 画面行错位，在快速运动场景中最明显")
print("3. 带宽下降 -> 整体清晰度降低，细节丢失")
print("4. 真实系统中三种损伤往往同时发生，互相叠加")
